In [1]:
import pandas as pd
import torch
import os
import sys
from tqdm import tqdm 
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_squared_error

# --- USER CHECK: Ensure python can find your 'polygraphpy' package ---
# If your notebook is inside the 'polygraphpy' folder, you might need to go up one level.
sys.path.append(os.path.abspath("..")) 

from polygraphpy.gnn.prediction import Prediction

def run_uq_analysis(validation_path, model_path, output_path, polymer_type, n_runs=100):
    """
    Runs the GNN model multiple times with Dropout enabled to quantify uncertainty.
    
    :param validation_path: Path to validation .pt files
    :param model_path: Path where the .pt model file is located
    :param output_path: Path where the resulting UQ csv files will be saved
    :param polymer_type: 'monomer' (for homopolymer) or 'copolymer'
    :param n_runs: Number of stochastic forward passes (default 100)
    """
    
    # 1. Initialize the existing Prediction class to load model and data
    print(f"Initializing loader for {polymer_type}...")
    
    # Check if paths exist to prevent immediate crashes
    if not os.path.exists(validation_path):
        raise FileNotFoundError(f"Validation path not found: {validation_path}")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path not found: {model_path}")

    runner = Prediction(validation_path, model_path, polymer_type)
    
    # 2. Extract Ground Truth (y) once
    ground_truths = []
    ground_truths = [graph.y.numpy()[0] for graph in runner.val_dataset]
    
    # Prepare dictionaries for data collection
    uq_results = {'y': ground_truths}
    error_metrics = {'mape': [], 'r2': [], 'mse': []}
    
    print(f"Starting {n_runs} stochastic runs for {polymer_type}...")
    
    # 3. Execution Loop
    for i in tqdm(range(n_runs), desc=f"Runs ({polymer_type})"):
        # CRITICAL: Switch model to train mode to enable Dropout for UQ
        runner.model.train()
        
        preds_run = []
        
        # Iterate over dataset
        for graph in runner.val_dataset:
            graph = graph.to(runner.device)
            batch = torch.zeros(graph.x.size(0), dtype=torch.long).to(runner.device)
            
            # Use no_grad to save memory, but keep Dropout active via model.train()
            with torch.no_grad():
                out = runner.model(graph.x, graph.edge_index, graph.edge_weight, batch)
                preds_run.append(out.detach().cpu().numpy()[0][0])
        
        # Store predictions for this run
        uq_results[f'run_{i}'] = preds_run
        
        # Calculate and store error metrics for this run
        mape = mean_absolute_percentage_error(ground_truths, preds_run) * 100
        r2 = r2_score(ground_truths, preds_run)
        mse = mean_squared_error(ground_truths, preds_run)
        
        error_metrics['mape'].append(round(mape, 5))
        error_metrics['r2'].append(round(r2, 5))
        error_metrics['mse'].append(round(mse, 5))

    # 4. Construct Final DataFrames
    df_uq = pd.DataFrame(uq_results)
    
    # Sort by ground truth 'y' to match the visualization needs
    df_uq = df_uq.sort_values(by='y').reset_index(drop=True)
    
    df_error = pd.DataFrame(error_metrics)

    # 5. Determine Filenames and Save
    if polymer_type == 'copolymer':
        name_suffix = 'copolymer'
    else:
        name_suffix = 'homopolymer'
        
    file_uq = os.path.join(output_path, f'results_uq_{name_suffix}.csv')
    file_err = os.path.join(output_path, f'results_uq_error_{name_suffix}.csv')
    
    df_uq.to_csv(file_uq, index=False)
    df_error.to_csv(file_err, index=False)
    
    print(f"Saved: {file_uq}")
    print(f"Saved: {file_err}")
    print("-" * 30)

In [2]:
# Define paths
# Verify these match your folder structure relative to this notebook
val_path_homo = '../polygraphpy/data/validation_data' 
model_path_homo = '../polygraphpy/data/gnn_output/'

val_path_copo = '../polygraphpy/data/validation_data_copoly'
model_path_copo = '../polygraphpy/data/gnn_output_copoly/'

out_path = '../polygraphpy/data/uq_analysis/'
os.makedirs(out_path, exist_ok=True)

# --- Run 1: Homopolymer/Monomer ---
try:
    run_uq_analysis(
        validation_path=val_path_homo, 
        model_path=model_path_homo, 
        output_path=out_path, 
        polymer_type='monomer', 
        n_runs=100
    )
except Exception as e:
    print(f"Error running Homopolymer analysis: {e}")

# --- Run 2: Copolymer ---
try:
    run_uq_analysis(
        validation_path=val_path_copo, 
        model_path=model_path_copo, 
        output_path=out_path, 
        polymer_type='copolymer', 
        n_runs=100
    )
except Exception as e:
    print(f"Error running Copolymer analysis: {e}")

Initializing loader for monomer...
Using device: cuda
Loading trained model.


/home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/libpyg.so: undefined symbol: _ZN2at4_ops11multinomial4callERKNS_6TensorElbSt8optionalINS_9GeneratorEE
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/torch_sparse/_hgt_sample_cuda.so: undefined symbol: _ZN2at4_ops11multinomial4callERKNS_6TensorElbSt8optionalINS_9GeneratorEE
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
/home/jgduarte/Documents/RA/Projects/3M/Po

GraphUNetModel(
  (unet): GraphUNet(56, 175, 175, depth=6, pool_ratios=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
  (lin1): Linear(in_features=175, out_features=175, bias=True)
  (lin2): Linear(in_features=175, out_features=175, bias=True)
  (lin3): Linear(in_features=175, out_features=175, bias=True)
  (output): Linear(in_features=175, out_features=1, bias=True)
)
Reading validation data.
Starting 100 stochastic runs for monomer...


Runs (monomer):   0%|          | 0/100 [00:00<?, ?it/s]/home/jgduarte/Documents/RA/Projects/3M/PolyGraphPy/.venv/lib/python3.13/site-packages/torch_geometric/utils/sparse.py:277: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  adj = torch.sparse_csr_tensor(
Runs (monomer): 100%|██████████| 100/100 [05:20<00:00,  3.21s/it]


Saved: ../polygraphpy/data/uq_analysis/results_uq_homopolymer.csv
Saved: ../polygraphpy/data/uq_analysis/results_uq_error_homopolymer.csv
------------------------------
Initializing loader for copolymer...
Using device: cuda
Loading trained model.
GraphUNetModel(
  (unet): GraphUNet(47, 175, 175, depth=6, pool_ratios=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
  (lin1): Linear(in_features=175, out_features=175, bias=True)
  (lin2): Linear(in_features=175, out_features=175, bias=True)
  (lin3): Linear(in_features=175, out_features=175, bias=True)
  (output): Linear(in_features=175, out_features=1, bias=True)
)
Reading validation data.
Starting 100 stochastic runs for copolymer...


Runs (copolymer): 100%|██████████| 100/100 [12:43<00:00,  7.64s/it]

Saved: ../polygraphpy/data/uq_analysis/results_uq_copolymer.csv
Saved: ../polygraphpy/data/uq_analysis/results_uq_error_copolymer.csv
------------------------------
